# Wine Quality Data Workflow
**Name:** Prof. Dr. Sc., Dr. Qeis Kamran  
**Dataset:** UCI Wine Quality (Red Wine) — [https://archive.ics.uci.edu/ml/datasets/wine+quality](https://archive.ics.uci.edu/ml/datasets/wine+quality)  
**Project:** Udacity MSc AI — AI Programming Foundations Project

This project builds a complete, reproducible data workflow using the UCI Red Wine Quality dataset. The dataset contains physicochemical properties of red wines (e.g., acidity, alcohol, pH) and a quality score rated by sommeliers. The goal is to clean, explore, and visualize the data to identify patterns that distinguish high-quality from low-quality wines.

---
## Notebook Structure
1. Setup & Imports
2. Data Ingestion
3. Data Cleaning
4. Exploratory Analysis
5. Visualizations
6. Summary & Interpretation

In [ ]:
# === SECTION 1: SETUP & IMPORTS ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✓ NumPy:', np.__version__)
print('✓ Pandas:', pd.__version__)
print('✓ All libraries imported successfully')

## Section 2: Data Ingestion

In [ ]:
# === SECTION 2: DATA INGESTION ===
df = pd.read_csv('wine.csv', sep=';')

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print()
df.head()

## Section 3: Data Cleaning

In [ ]:
# === SECTION 3: DATA CLEANING ===

def remove_duplicates(dataframe):
    """
    Remove duplicate rows from the dataset.
    
    Duplicate records can distort statistical summaries and model training
    by artificially inflating the weight of repeated observations.
    This function identifies and drops all duplicate rows, then reports
    how many were removed.
    
    Parameters:
        dataframe (pd.DataFrame): Input dataset.
    Returns:
        pd.DataFrame: Dataset with duplicates removed.
    """
    n_before = len(dataframe)
    cleaned = dataframe.drop_duplicates()
    n_removed = n_before - len(cleaned)
    print(f'  Duplicates removed: {n_removed} (from {n_before} to {len(cleaned)} rows)')
    return cleaned.reset_index(drop=True)


def handle_outliers(dataframe, columns, method='clip', threshold=3.0):
    """
    Detect and handle outliers using z-score clipping.
    
    Extreme outliers in physicochemical measurements can be caused by
    measurement error. This function clips values beyond a z-score threshold
    to the nearest boundary, preserving the row while reducing distortion.
    Clipping is preferred over deletion to maintain dataset size (McKinney, 2022).
    
    Parameters:
        dataframe (pd.DataFrame): Input dataset.
        columns (list): Column names to check for outliers.
        method (str): 'clip' clips to boundary; 'remove' drops outlier rows.
        threshold (float): Z-score threshold (default 3.0 = 99.7% of normal dist).
    Returns:
        pd.DataFrame: Dataset with outliers handled.
    """
    df_clean = dataframe.copy()
    total_flagged = 0
    for col in columns:
        z_scores = np.abs((df_clean[col] - df_clean[col].mean()) / df_clean[col].std())
        flagged = (z_scores > threshold).sum()
        total_flagged += flagged
        if method == 'clip':
            lower = df_clean[col].mean() - threshold * df_clean[col].std()
            upper = df_clean[col].mean() + threshold * df_clean[col].std()
            df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    print(f'  Outliers handled: {total_flagged} values clipped across {len(columns)} columns')
    return df_clean


# Apply cleaning functions
print('Applying cleaning pipeline:')
df_clean = remove_duplicates(df)
numeric_cols = [c for c in df_clean.columns if c != 'quality']
df_clean = handle_outliers(df_clean, numeric_cols)

# Check for missing values
missing = df_clean.isnull().sum().sum()
print(f'  Missing values: {missing}')
print(f'\nFinal dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print('\n✓ Data cleaning complete')

## Section 4: Exploratory Analysis

In [ ]:
# === SECTION 4: EXPLORATORY ANALYSIS ===

def explore_wine_quality(dataframe):
    """
    Perform comprehensive exploratory analysis of the wine quality dataset.
    
    Computes summary statistics, quality distribution, and correlation
    between physicochemical properties and quality scores.
    Returns a correlation series for use in visualizations.
    
    Parameters:
        dataframe (pd.DataFrame): Cleaned wine dataset.
    Returns:
        pd.Series: Correlation of each feature with quality score.
    """
    print('=== SUMMARY STATISTICS ===')
    print(dataframe.describe().round(2))
    
    print('\n=== QUALITY DISTRIBUTION ===')
    quality_counts = dataframe['quality'].value_counts().sort_index()
    for score, count in quality_counts.items():
        bar = '█' * (count // 5)
        print(f'  Quality {score}: {count:4d} wines  {bar}')
    
    print('\n=== CORRELATION WITH QUALITY (top 5) ===')
    corr = dataframe.corr()['quality'].drop('quality').sort_values(key=abs, ascending=False)
    for feat, val in corr.head(5).items():
        direction = '↑' if val > 0 else '↓'
        print(f'  {feat:<30} {direction} r={val:.3f}')
    
    print('\n=== HIGH vs LOW QUALITY COMPARISON ===')
    df['quality_label'] = pd.cut(dataframe['quality'], bins=[0,5,10],
                                  labels=['Low (3-5)', 'High (6-8)'])
    comparison = dataframe.groupby(df['quality_label'])[['alcohol','volatile acidity','sulphates']].mean().round(3)
    print(comparison)
    
    return corr

# Run EDA
correlations = explore_wine_quality(df_clean)

## Section 5: Visualizations

In [ ]:
# === FIGURE 1: Quality Distribution ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: count plot
quality_counts = df_clean['quality'].value_counts().sort_index()
colors = ['#d32f2f' if q <= 5 else '#388e3c' for q in quality_counts.index]
bars = axes[0].bar(quality_counts.index, quality_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Figure 1A: Distribution of Wine Quality Scores', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Quality Score (Sommelier Rating)', fontsize=11)
axes[0].set_ylabel('Number of Wines', fontsize=11)
axes[0].set_xticks(quality_counts.index)
for bar, count in zip(bars, quality_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].legend(handles=[
    plt.Rectangle((0,0),1,1,color='#d32f2f',label='Low quality (3-5)'),
    plt.Rectangle((0,0),1,1,color='#388e3c',label='High quality (6-8)')
], loc='upper right', fontsize=10)

# Right: pie
low = (df_clean['quality'] <= 5).sum()
high = (df_clean['quality'] >= 6).sum()
axes[1].pie([low, high], labels=[f'Low quality\n(score ≤5)\n{low} wines',
                                   f'High quality\n(score ≥6)\n{high} wines'],
            colors=['#ef9a9a','#a5d6a7'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Figure 1B: Low vs High Quality Split', fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('figure1_quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# === FIGURE 2: Feature Correlations with Quality ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: correlation bar chart
corr_vals = df_clean.corr()['quality'].drop('quality').sort_values()
colors = ['#ef5350' if v < 0 else '#42a5f5' for v in corr_vals]
axes[0].barh(corr_vals.index, corr_vals.values, color=colors, edgecolor='white')
axes[0].axvline(x=0, color='black', linewidth=1.5, linestyle='--', alpha=0.5)
axes[0].set_title('Figure 2A: Feature Correlation with Wine Quality', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Pearson Correlation Coefficient (r)', fontsize=11)
axes[0].set_ylabel('Physicochemical Feature', fontsize=11)
axes[0].legend(handles=[
    plt.Rectangle((0,0),1,1,color='#42a5f5',label='Positive correlation'),
    plt.Rectangle((0,0),1,1,color='#ef5350',label='Negative correlation')
], fontsize=10)

# Right: alcohol vs quality boxplot
quality_groups = [df_clean[df_clean['quality']==q]['alcohol'].values
                  for q in sorted(df_clean['quality'].unique())]
bp = axes[1].boxplot(quality_groups, labels=sorted(df_clean['quality'].unique()),
                      patch_artist=True, medianprops={'color':'black','linewidth':2})
colors_box = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(quality_groups)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Figure 2B: Alcohol Content by Quality Score', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Quality Score', fontsize=11)
axes[1].set_ylabel('Alcohol (% by volume)', fontsize=11)

plt.tight_layout()
plt.savefig('figure2_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# === FIGURE 3: Heatmap + Acidity vs Quality ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: correlation heatmap
key_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid',
                 'pH', 'fixed acidity', 'quality']
corr_matrix = df_clean[key_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
            annot_kws={'size': 9})
axes[0].set_title('Figure 3A: Correlation Heatmap\n(Key Features)', fontsize=13, fontweight='bold', pad=12)
axes[0].tick_params(axis='x', rotation=35)

# Right: volatile acidity vs quality scatter
quality_colors = {3:'#b71c1c',4:'#e53935',5:'#ff7043',
                  6:'#66bb6a',7:'#2e7d32',8:'#1b5e20'}
for q in sorted(df_clean['quality'].unique()):
    subset = df_clean[df_clean['quality']==q]
    axes[1].scatter(subset['volatile acidity'], subset['alcohol'],
                    c=quality_colors.get(q,'gray'), label=f'Quality {q}',
                    alpha=0.6, s=40, edgecolors='white', linewidth=0.3)
axes[1].set_title('Figure 3B: Alcohol vs Volatile Acidity\n(coloured by Quality)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Volatile Acidity (g/dm³)', fontsize=11)
axes[1].set_ylabel('Alcohol (% by volume)', fontsize=11)
axes[1].legend(title='Quality', fontsize=9, title_fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('figure3_heatmap_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## Section 6: Summary and Interpretation

### What I Learned from the Dataset

The UCI Red Wine Quality dataset reveals clear physicochemical patterns that distinguish high-quality from low-quality wines. The most consistent finding across all three visualizations is that **alcohol content** is the strongest positive predictor of quality (r ≈ +0.48), while **volatile acidity** is the strongest negative predictor (r ≈ −0.39). This aligns with established oenological knowledge: higher alcohol indicates more complete fermentation and ripe grapes, while elevated volatile acidity (acetic acid) signals spoilage or poor fermentation control.

### Interesting Patterns and Insights

- **Figure 1** shows that the quality distribution is strongly concentrated at scores 5 and 6 (approximately 70% of all wines), with very few extreme scores (3 or 8). This class imbalance would be a significant challenge for any future machine learning model.
- **Figure 2B** clearly shows that high-quality wines (scores 7-8) consistently have higher median alcohol content and less variance than low-quality wines.
- **Figure 3B** reveals a visual separation: high-quality wines (green) cluster toward high alcohol / low volatile acidity, while low-quality wines (red) cluster toward low alcohol / high volatile acidity.
- **Sulphates** showed a moderate positive correlation with quality, likely because sulphates act as preservatives that protect wine from oxidation.

### Limitations and Assumptions

- The dataset represents only **red wine** (Vinho Verde variety from Portugal) — findings may not generalize to other wine types or regions.
- Quality scores are **subjective sommelier ratings** — inter-rater reliability is not reported, introducing measurement uncertainty.
- The **class imbalance** (very few wines scored 3, 4, or 8) means that any future classifier would need resampling strategies.
- Outlier **clipping** (z > 3.0) was used rather than deletion to preserve dataset size, which may slightly compress extreme values.

### What Was Surprising

The relatively weak correlation between **pH** and quality (r ≈ −0.06) was unexpected, as pH is often discussed as a key wine quality factor. This suggests that within the range of red wines, pH variation alone does not strongly predict sommelier scores — the combination of multiple features (alcohol, acidity, sulphates) matters more than any single measurement.

In [ ]:
# === Generate requirements.txt ===
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                        capture_output=True, text=True)
with open('requirements.txt', 'w') as f:
    f.write(result.stdout)
print('✓ requirements.txt generated')
print('\nKey packages:')
for line in result.stdout.split('\n'):
    if any(p in line.lower() for p in ['numpy','pandas','matplotlib','seaborn']):
        print(f'  {line}')